# Allstate · Bloc 2 — Porteur P3
**2.3 Le plancher de contrôle** · **2.6 Les comparaisons**

Kaggle : *Add Input* → votre jeu de données (`train.csv`, `a_estimer.csv`) · *Accelerator : None* · Internet off.
Métrique : **MAE** (erreur absolue), cible `loss`.

Dépendances d'équipe (à brancher, voir cellule 2) : découpage **2.1** (P1) et barre de bruit **2.2** (P2).
Sortie pour **2.7 Le tableau d'essai** : `/kaggle/working/sorties/comparaisons_p3.csv`.

In [1]:
import glob, os, time, json, warnings
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error as MAE
warnings.filterwarnings("ignore")

def trouver(nom):
    for p in glob.glob(f"/kaggle/input/**/{nom}", recursive=True) + glob.glob(f"./**/{nom}", recursive=True):
        return p
    raise FileNotFoundError(nom)

train = pd.read_csv(trouver("train.csv"))
test  = pd.read_csv(trouver("a_estimer.csv"))
CIBLE = "loss"
CATS  = [c for c in train.columns if c.startswith("cat")]
CONTS = [c for c in train.columns if c.startswith("cont")]
y = train[CIBLE].to_numpy()
print(train.shape, test.shape, len(CATS), "cat /", len(CONTS), "cont")
os.makedirs("/kaggle/working/sorties", exist_ok=True)

(150654, 132) (37664, 131) 116 cat / 14 cont


In [2]:
# --- Identité de la séance (à remplir) ---------------------------------------
SEANCE           = "Séance 2"                      # numéro ou date de la séance
NOM              = "Faozane "                        # votre nom
PORTEUR          = "P3"                       # votre numéro de porteur
ETAPES_ASSIGNEES = ["2.6", "2.3",  ]   # vos étapes assignées

# Mémorisé dans sorties/identite_p3.json, pour identifier l'auteur du livrable
RESULTATS = {"identite": dict(seance=SEANCE, nom=NOM.strip(), porteur=PORTEUR, etapes=ETAPES_ASSIGNEES)}
os.makedirs("/kaggle/working/sorties", exist_ok=True) if os.path.exists("/kaggle/working") else os.makedirs("sorties", exist_ok=True)
_sortie = "/kaggle/working/sorties" if os.path.exists("/kaggle/working") else "sorties"
with open(f"{_sortie}/identite_p3.json", "w", encoding="utf-8") as f:
    json.dump(RESULTATS, f, indent=2, ensure_ascii=False)
print(f"{SEANCE} · {NOM.strip()} · porteur {PORTEUR} · étapes assignées : {', '.join(ETAPES_ASSIGNEES)}")

Séance 2 · Faozane · porteur P3 · étapes assignées : 2.6, 2.3


## 0. Branchement sur le travail des autres
**À REMPLACER** par le découpage retenu en **2.1** (P1) et la barre de bruit mesurée en **2.2** (P2).
Par défaut : 5 plis stratifiés sur les déciles de `loss` (cible continue → on stratifie sur des tranches), graine fixe.

In [3]:
SEED, N_PLIS = 42, 5          # <- 2.1 (P1) : remplacer si le découpage retenu diffère
BARRE_DE_BRUIT = None         # <- 2.2 (P2) : écart de MAE attribuable au hasard ; None = écart-type entre plis

def plis():
    tranches = pd.qcut(y, 10, labels=False, duplicates="drop")
    return list(StratifiedKFold(N_PLIS, shuffle=True, random_state=SEED).split(train, tranches))

PLIS = plis()
print([ (len(a), len(b)) for a, b in PLIS ])

[(120523, 30131), (120523, 30131), (120523, 30131), (120523, 30131), (120524, 30130)]


## 2.3 Le plancher de contrôle
Une estimation **sans modèle** : une seule valeur annoncée pour tous les sinistres. Pour la MAE, l'optimum théorique est la **médiane** ; on le **mesure** au lieu de le supposer : chaque constante est ajustée sur le pli d'entraînement, évaluée sur le pli de validation.

In [4]:
def constantes(v):
    return {
        "moyenne": v.mean(),
        "médiane": np.median(v),
        "moyenne géométrique": np.expm1(np.log1p(v).mean()),
        "quantile 40 %": np.quantile(v, .40),
        "quantile 60 %": np.quantile(v, .60),
    }

lignes = []
for k, (a, b) in enumerate(PLIS):
    for nom, c in constantes(y[a]).items():
        lignes.append({"pli": k, "constante": nom, "valeur": c,
                       "MAE": MAE(y[b], np.full(len(b), c)),
                       "RMSE": np.sqrt(np.mean((y[b] - c) ** 2))})
plancher = pd.DataFrame(lignes)
resume = (plancher.groupby("constante")
          .agg(valeur=("valeur", "mean"), MAE=("MAE", "mean"), MAE_std=("MAE", "std"), RMSE=("RMSE", "mean"))
          .sort_values("MAE"))
display(resume.round(2))

,valeur,MAE,MAE_std,RMSE
constante,,,,
médiane,2115.56,1810.49,8.74,3056.89
moyenne géométrique,2177.22,1811.29,8.77,3038.83
quantile 40 %,1703.63,1850.21,8.64,3205.52
quantile 60 %,2647.25,1866.11,8.78,2940.27
moyenne,3038.92,1968.14,8.22,2914.08


In [5]:
# Vérification empirique : balayage de la constante sur tout train -> le minimum de MAE tombe-t-il sur la médiane ?
grille = np.quantile(y, np.linspace(.30, .70, 81))
courbe = [np.mean(np.abs(y - c)) for c in grille]
c_opt = grille[int(np.argmin(courbe))]
print(f"constante minimisant la MAE (balayage) : {c_opt:,.2f}  | médiane : {np.median(y):,.2f}  | moyenne : {y.mean():,.2f}")

# Coût asymétrique : sous- vs sur-estimation avec la médiane
c = np.median(y); e = y - c
print(f"sous-estimations (y > c) : {np.mean(e > 0):.1%} des sinistres, erreur moyenne {e[e>0].mean():,.0f}")
print(f"sur-estimations  (y < c) : {np.mean(e < 0):.1%} des sinistres, erreur moyenne {(-e[e<0]).mean():,.0f}")

PLANCHER_NOM = resume.index[0]
PLANCHER_MAE, PLANCHER_STD = resume.loc[PLANCHER_NOM, "MAE"], resume.loc[PLANCHER_NOM, "MAE_std"]
print(f"\n>>> PLANCHER DE CONTRÔLE : {PLANCHER_NOM} = {resume.loc[PLANCHER_NOM,'valeur']:,.2f} · MAE = {PLANCHER_MAE:,.2f} ± {PLANCHER_STD:,.2f}")
json.dump({"constante": PLANCHER_NOM, "valeur": float(resume.loc[PLANCHER_NOM, "valeur"]),
           "MAE": float(PLANCHER_MAE), "MAE_std_plis": float(PLANCHER_STD), "seed": SEED, "n_plis": N_PLIS},
          open("/kaggle/working/sorties/plancher_p3.json", "w"), indent=2)

constante minimisant la MAE (balayage) : 2,115.57  | médiane : 2,115.57  | moyenne : 3,038.92
sous-estimations (y > c) : 50.0% des sinistres, erreur moyenne 2,734
sur-estimations  (y < c) : 50.0% des sinistres, erreur moyenne 887

>>> PLANCHER DE CONTRÔLE : médiane = 2,115.56 · MAE = 1,810.49 ± 8.74


**À consigner dans la note** : la médiane bat la moyenne en MAE parce que `loss` est très asymétrique (queue lourde, jusqu'à ~121 000) ; la moyenne est tirée vers le haut par quelques gros sinistres. La sous-estimation des gros sinistres pèse plus en volume d'erreur que la sur-estimation des petits.

## 2.6 Les comparaisons
Tous les candidats passent **sur les mêmes plis**, avec **la même métrique**, face au plancher. Un gain n'est retenu que s'il dépasse la **barre de bruit**.

In [6]:
# Encodages (sans cible -> pas de fuite ; catégories vues sur train + a_estimer)
tout = pd.concat([train[CATS], test[CATS]])
ordinal = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1).fit(tout)
Xo = train[CATS + CONTS].copy()
Xo[CATS] = ordinal.transform(train[CATS]).astype(int)
card = tout.nunique()
masque_cat = [bool(card[c] <= 255) if c in CATS else False for c in Xo.columns]   # HGB : 255 modalités max

ohe = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), CATS),
    ("num", StandardScaler(), CONTS)])
ohe.fit(pd.concat([train, test])[CATS + CONTS])
Xl = ohe.transform(train[CATS + CONTS])
print("one-hot :", Xl.shape, "| catégorielles natives HGB :", sum(masque_cat), "/", len(CATS))

one-hot : (150654, 868) | catégorielles natives HGB : 115 / 116


In [7]:
def hgb(**kw):
    return HistGradientBoostingRegressor(categorical_features=masque_cat, max_iter=300, learning_rate=0.08,
                                         early_stopping=False, random_state=SEED, **kw)

CANDIDATS = {
  # nom : (X, fabrique, transformation cible)
  "ridge (one-hot, log cible)":      (Xl, lambda: Ridge(alpha=10.0),                 "log"),
  "HGB perte absolue (brute)":       (Xo, lambda: hgb(loss="absolute_error"),        None),
  "HGB perte carrée (log cible)":    (Xo, lambda: hgb(loss="squared_error"),         "log"),
  "HGB perte absolue (log cible)":   (Xo, lambda: hgb(loss="absolute_error"),        "log"),
}

def evaluer(nom, X, fab, tr):
    t0, maes = time.time(), []
    for a, b in PLIS:
        yt = np.log1p(y[a]) if tr == "log" else y[a]
        m = fab().fit(X.iloc[a] if hasattr(X, "iloc") else X[a], yt)
        p = m.predict(X.iloc[b] if hasattr(X, "iloc") else X[b])
        p = np.expm1(p) if tr == "log" else p
        maes.append(MAE(y[b], np.clip(p, 0, None)))
    return {"essai": nom, "MAE": np.mean(maes), "MAE_std": np.std(maes, ddof=1), "MAE_plis": np.round(maes, 2).tolist(),
            "gain_vs_plancher_%": 100 * (1 - np.mean(maes) / PLANCHER_MAE), "secondes": round(time.time() - t0)}

resultats = [{"essai": f"plancher : {PLANCHER_NOM}", "MAE": PLANCHER_MAE, "MAE_std": PLANCHER_STD,
              "MAE_plis": plancher[plancher.constante == PLANCHER_NOM].MAE.round(2).tolist(),
              "gain_vs_plancher_%": 0.0, "secondes": 0}]
for nom, (X, fab, tr) in CANDIDATS.items():
    r = evaluer(nom, X, fab, tr); resultats.append(r)
    print(f"{nom:34s} MAE {r['MAE']:>9,.2f} ± {r['MAE_std']:.2f} | gain {r['gain_vs_plancher_%']:5.1f} % | {r['secondes']} s")

ridge (one-hot, log cible)         MAE  1,249.54 ± 8.85 | gain  31.0 % | 72 s
HGB perte absolue (brute)          MAE  1,159.58 ± 5.85 | gain  36.0 % | 132 s
HGB perte carrée (log cible)       MAE  1,152.43 ± 7.57 | gain  36.3 % | 123 s
HGB perte absolue (log cible)      MAE  1,153.12 ± 6.66 | gain  36.3 % | 132 s


In [8]:
comp = pd.DataFrame(resultats)
bruit = BARRE_DE_BRUIT if BARRE_DE_BRUIT is not None else comp.MAE_std.max()
comp["écart_abs"] = PLANCHER_MAE - comp.MAE
comp["> bruit ?"] = comp["écart_abs"] > bruit
comp = comp.sort_values("MAE").reset_index(drop=True)
display(comp.round(2))
print(f"barre de bruit utilisée : {bruit:,.2f}  ({'2.2 P2' if BARRE_DE_BRUIT is not None else 'écart-type entre plis, provisoire'})")

meilleur = comp.iloc[0]
print(f"\nMeilleur : {meilleur.essai} — MAE {meilleur.MAE:,.2f}, soit {meilleur['gain_vs_plancher_%']:.1f} % mieux que le plancher ({PLANCHER_NOM}).")
comp.to_csv("/kaggle/working/sorties/comparaisons_p3.csv", index=False)

,essai,MAE,MAE_std,MAE_plis,gain_vs_plancher_%,secondes,écart_abs,> bruit ?
0,HGB perte carrée (log cible),1152.43,7.57,"[1157.48, 1163.07, 1145.61, 1146.53, 1149.48]",36.35,123,658.05,True
1,HGB perte absolue (log cible),1153.12,6.66,"[1156.43, 1163.15, 1146.63, 1149.42, 1149.97]",36.31,132,657.36,True
2,HGB perte absolue (brute),1159.58,5.85,"[1160.71, 1169.25, 1155.41, 1155.06, 1157.48]",35.95,132,650.90,True
3,"ridge (one-hot, log cible)",1249.54,8.85,"[1249.01, 1264.34, 1248.33, 1241.13, 1244.88]",30.98,72,560.95,True
4,plancher : médiane,1810.49,8.74,"[1818.13, 1820.12, 1809.48, 1805.59, 1799.1]",0.00,0,0.00,False


barre de bruit utilisée : 8.85  (écart-type entre plis, provisoire)

Meilleur : HGB perte carrée (log cible) — MAE 1,152.43, soit 36.3 % mieux que le plancher (médiane).


**Transmission à P1 (2.7 Le tableau d'essai)** : `comparaisons_p3.csv` contient, pour chaque essai, la MAE par pli, la moyenne, l'écart-type, le gain relatif au plancher et le verdict face à la barre de bruit. Les essais qui ne passent pas la barre restent dans le tableau : le journal consigne les échecs.

**Pour le rendu (dépôt)** : copier les fonctions de `2.3` et `2.6` dans le `pipeline.py` ; les candidats se changent dans le dictionnaire `CANDIDATS` (une cellule, c'est la mutation « réparable » demandée).